# Practice 2. Did the new prompt make the system better?

**ML in Industry 2026 · Topic 2. Evaluating LLM systems**

## After practice 1

The system reads one maintenance log entry and returns one record that follows a JSON Schema.

```
21.07.25, агрегат остановлен, M-310: устранение неисправности, обнаружено низкое сопротивление изоляции, 6 hrs, Гусев К.
  ↓
{"equipment_id": "M-310", "equipment_type": "electric_motor", "work_type": "corrective_repair",
 "failure_mode": "electrical_fault", "replaced_parts": [], "duration_hours": 6, "work_date": "2025-07-21",
 "downtime": true, "defect_summary": "Low insulation resistance detected.", "evidence": [...]}
```

- The log has 300 synthetic entries in Russian, English or both, with typos. Every entry is labelled, because the generator chose the facts first and wrote the text second.
- Eight typed fields are compared with the labels. The course model gets 99.8% field accuracy on `eval_50`.
- The free-text field `defect_summary` has no label and was not evaluated in practice 1.

## Plan

Change the extraction prompt for `defect_summary` and measure the effect.

| Part | Block | Run | Measure |
|------|-------|-----|---------|
| A. The system | A1 The change | prompt v1 and v2 | |
| | A2 Typed fields | v1 and v2, 3 runs each, 100 entries | field accuracy, all fields right, eval_50 and holdout |
| | A3 Free text | the same runs, graded by an LLM judge | faithful summaries, interval, paired test |
| | A4 Gate | v2 against v1 | pass or fail |
| B. The judge | B1 Human labels | 213 summaries from practice 1 | PASS or FAIL with a reason |
| | B2 Judges | 2 judge prompts, thinking off and on | agreement, kappa, caught failures |
| | B3 Disagreements | judge and person | causes of disagreement |
| | B4 Few labels | judge on all, 40 human labels | PPI estimate with an interval |

| Split | Size | Role |
|-------|------|------|
| `eval_50` | 50 | golden set, used to build practice 1 |
| `holdout_50` | 50 | entries nobody looked at, used to confirm results |

The notebook needs the course model token in `.env`. The first run makes about 1 600 calls and takes 10–20 minutes. Results are stored in `runs/` and reused on the next run.

In [1]:
import os

import pandas as pd

from plant_assistant import PROJECT_ROOT, RUNS_DIR, prompts
from plant_assistant.config import get_profile, load_env
from plant_assistant.evals.cases import fingerprint, load_cases
from plant_assistant.llm import LLMClient

pd.set_option("display.max_colwidth", 110)
load_env()
course = LLMClient(get_profile("course"))
print("course model:", "OK" if course.chat([{"role": "user", "content": "Reply with one word: OK"}]).ok else "NOT AVAILABLE, check .env")
JUDGE = "course-thinking"  # binary judge with thinking on, it has the highest kappa in part B


golden = [c for c in load_cases(PROJECT_ROOT / "evals" / "cases_v1.jsonl") if c.source == "log"]
holdout = load_cases(PROJECT_ROOT / "evals" / "holdout_v1.jsonl")
cases = golden + holdout
print(f"eval_50: {len(golden)} entries ({fingerprint(golden)}), holdout_50: {len(holdout)} entries ({fingerprint(holdout)})")

course model: OK
eval_50: 50 entries (9216276ceb), holdout_50: 50 entries (430edaaf67)


# Part A. The system, prompt v1 and prompt v2

## A1. The change

In practice 1 the model often added a cause or a location that the entry does not state. Examples labelled by a person.

In [2]:
from plant_assistant.evals.analysis import labelled_pool

pool = labelled_pool()
pool[pool["rule"] == "cause"][["text", "summary", "note"]].head(4)

,text,summary,note
0,"2025-08-09 P-202A repair after failure, unit stopped, blockage found, filter replaced, 12 h, technician Гу...",Blockage found causing failure,the entry does not say that the blockage caused the failure
6,"2025-08-09 P-202A repair after failure, unit stopped, blockage found, filter replaced, 12 h, technician Гу...",unit stopped due to blockage,the entry does not say that the unit stopped because of the blockage
34,2025-01-03 F-230 PM stopped 4 hrs shaft misalign,Shaft misalignment caused the fan to stop.,"the fan was stopped for preventive maintenance, not because of the misalignment"
36,21.11.2025 P-202A repair after failure unit stopped 12h plugged strainer,Strainer plugged causing unit failure,the entry does not say that the strainer caused the failure


Prompt v1 is the prompt of practice 1. Prompt v2 adds one rule at the end of the system prompt.

In [3]:
v1 = prompts.extraction_messages("<entry>")[0]["content"]
v2 = prompts.extraction_messages("<entry>", summary_rule=True)[0]["content"]
print("v1 ends with\n  ..." + v1[-160:])
print("\nv2 adds\n  " + v2[len(v1):].strip())

v1 ends with
  ...topped for the work, false if it kept running. null if not stated.
- evidence: Short exact quotes copied from the source text that support the extracted values.

v2 adds
  Rule for defect_summary: describe only what the entry says about the defect, in English. Do not add a cause, a location or a part that the entry does not state. If the entry says that no problem was found, use null.


Prompt v2 on five entries, next to prompt v1.

In [4]:
from plant_assistant.evals.harness import load_results, run_cases

five = golden[:5]
v1_five = run_cases("five-v1", course, five, mode="structured", progress=False)
v2_five = run_cases("five-v2", course, five, mode="structured-v2", progress=False)
pd.DataFrame([{"entry": c.text, "v1 summary": (v1_five[c.case_id]["record"] or {}).get("defect_summary"),
               "v2 summary": (v2_five[c.case_id]["record"] or {}).get("defect_summary")} for c in five])

,entry,v1 summary,v2 summary
0,"2025-08-09 P-202A repair after failure, unit stopped, blockage found, filter replaced, 12 h, technician Гу...",Blockage found in the unit.,Blockage found in the unit.
1,"21.07.25, агрегат остановлен, M-310: устранение неисправности, обнаружено низкое сопротивление изоляции, 6...",Low insulation resistance detected.,Low insulation resistance detected.
2,"FV305, внеплановый ремонт 30.06.2024, обнаружена коррозия седла, замеенна прокладка, простой, 6 hrs, Захар...",Corrosion of the valve seat was detected.,Corrosion of the seat was detected.
3,P-102A 31.05.2026 продувка и чистка засор найден заменён фильтр 30мин не останавливали,Blockage found during cleaning.,Blockage found during cleaning.
4,2025-01-03 F-230 PM stopped 4 hrs shaft misalign,Shaft misalignment detected during preventive maintenance.,Shaft misalignment


The blocks below use both prompts, 3 runs each, on 100 entries (`eval_50` and `holdout_50`), course model.

## A2. Typed fields

> **Predict.** Will the typed fields stay the same with prompt v2?

In [5]:
from plant_assistant.evals.analysis import experiment_table
from plant_assistant.evals.runs import extraction_runs, run_judges

RUNS = extraction_runs(cases)  # 6 runs of 100 entries, about 5 minutes on the first run
run_judges(RUNS, profile=JUDGE)  # the judge grades every summary, about 3 minutes on the first run
exp = experiment_table(cases, JUDGE)
exp[["prompt", "setup", "valid, %", "field accuracy, %", "all fields right, %"]]

,prompt,setup,"valid, %","field accuracy, %","all fields right, %"
0,v1,course-v1-r1,98.0,96.8,90.0
1,v1,course-v1-r2,98.0,96.8,90.0
2,v1,course-v1-r3,98.0,96.6,89.0
3,v2,course-v2-r1,98.0,96.4,89.0
4,v2,course-v2-r2,98.0,96.6,90.0
5,v2,course-v2-r3,98.0,96.6,90.0


**Result.** Both prompts give about 96.6% field accuracy and 89–90% of entries with all fields right. Runs of one prompt differ a little even at temperature 0.

The same runs on `eval_50` and on the holdout.

In [6]:
from plant_assistant.evals.harness import score_cases, summary_row

rows = []
for run in ["course-v1-r1", "course-v2-r1"]:
    for part, cs in (("eval_50", golden), ("holdout_50", holdout)):
        rows.append({**summary_row(score_cases(cs, load_results(run)), run), "part": part})
pd.DataFrame(rows)[["setup", "part", "field accuracy, %", "all fields right, %"]]

,setup,part,"field accuracy, %","all fields right, %"
0,course-v1-r1,eval_50,99.5,96.0
1,course-v1-r1,holdout_50,94.0,84.0
2,course-v2-r1,eval_50,99.8,98.0
3,course-v2-r1,holdout_50,93.0,80.0


**Result.** All fields are right in 96–98% of `eval_50` entries and in 80–84% of holdout entries. `eval_50` is saturated, because practice 1 was tuned on it.

Error categories over the 100 entries. Invented means the text does not state the value. Missed means the text states it.

In [7]:
from plant_assistant.evals.errors import error_counts, error_table

pd.concat({run: error_counts(error_table(cases, load_results(run))) for run in ["course-v1-r1", "course-v2-r1"]})

category                     invented  missed  needs_review  wrong value  \
             field                                                         
course-v1-r1 work_type              0       0             0            6   
             <record>               0       0             2            0   
             equipment_id           0       1             0            0   
             equipment_type         0       0             0            1   
             replaced_parts         1       0             0            0   
             downtime               1       0             0            0   
course-v2-r1 work_type              0       0             0            6   
             equipment_id           0       3             0            0   
             equipment_type         0       0             0            3   
             <record>               0       0             2            0   
             replaced_parts         1       0             0            0   

category                     total  
             field                  
course-v1-r1 work_type           6  
             <record>            2  
             equipment_id        1  
             equipment_type      1  
             replaced_parts      1  
             downtime            1  
course-v2-r1 work_type           6  
             equipment_id        3  
             equipment_type      3  
             <record>            2  
             replaced_parts      1

**Result.** The most frequent typed error is a wrong `work_type` for both prompts.

## A3. Free text

Code checks first. English, at most 15 words, no string "null", null when no defect was found.

In [8]:
from collections import Counter

from plant_assistant.evals.checks import check_summary

for run in ["course-v1-r1", "course-v2-r1"]:
    res = load_results(run)
    problems = Counter(p for c in cases for p in check_summary((res[c.case_id]["record"] or {}).get("defect_summary"),
                                                               c.expected.get("failure_mode")))
    print(run, dict(problems) or "no code problems")

course-v1-r1 no code problems
course-v2-r1 no code problems


Both prompts pass the code checks. An LLM judge decides whether each summary says only what the entry says. Part B tests this judge against a person (kappa 0.82).

> **Predict.** Will prompt v2 give more faithful summaries? By how many points?

In [9]:
exp[["prompt", "setup", "summary faithful, %", "faithful 95% interval"]]

,prompt,setup,"summary faithful, %",faithful 95% interval
0,v1,course-v1-r1,91.0,84–95
1,v1,course-v1-r2,93.0,86–97
2,v1,course-v1-r3,91.0,84–95
3,v2,course-v2-r1,97.0,92–99
4,v2,course-v2-r2,98.0,93–99
5,v2,course-v2-r3,98.0,93–99


**Result.** 91–93% of summaries are faithful with v1 and 97–98% with v2, in every run. The intervals overlap a little, so the prompts are compared on the same entries with the McNemar test.

In [10]:
from plant_assistant.evals.analysis import experiment_compare, experiment_compare_pooled

pd.DataFrame([experiment_compare(cases, JUDGE, i) for i in range(3)] + [experiment_compare_pooled(cases, JUDGE)])

,A,B,n,A right,B right,B fixed,B broke,p_value,decision
0,course-v1-r1,course-v2-r1,100,91,97,6,0,0.0312,B is better
1,course-v1-r2,course-v2-r2,100,93,98,5,0,0.0625,no detectable difference
2,course-v1-r3,course-v2-r3,100,91,98,7,0,0.0156,B is better
3,"v1, majority of 3 runs","v2, majority of 3 runs",100,92,98,6,0,0.0312,B is better


**Result.** v2 fixed 5–7 summaries and broke none. Runs 1 and 3 give p < 0.05, run 2 gives p = 0.063. With few changed entries one entry moves the p-value a lot. All three runs together give p = 0.031.

Entries needed for a reliable decision.

In [11]:
from plant_assistant.evals.stats import n_independent, n_paired

r = experiment_compare(cases, JUDGE, 0)
p1, p2, changed = r["A right"] / r["n"], r["B right"] / r["n"], (r["B fixed"] + r["B broke"]) / r["n"]
print(f"v1 {p1:.0%} -> v2 {p2:.0%}, changed cases {changed:.0%}")
print(f"separate samples {n_independent(p1, p2)} entries per version, paired {n_paired(abs(p2 - p1), changed)} entries")

v1 91% -> v2 97%, changed cases 6%
separate samples 243 entries per version, paired 129 entries


## A4. Gate

The gate checks that the candidate is not worse than the baseline on the same entries. It fails when the candidate is significantly worse or breaks more than 2 entries.

In [12]:
from plant_assistant.evals.gate import main as gate

exit_codes = [gate(["course-v1-r1", "course-v2-r1", "--metric", "all_fields_right"])]
print()
exit_codes.append(gate(["local-structured", "local-small-structured"]))  # 2B model instead of 4B, from practice 1
print("exit codes", exit_codes)

course-v2-r1 vs course-v1-r1 on eval_50, metric all_fields_right
  baseline right 48/50, candidate right 49/50, fixed 1, broke 0, p = 1.0  ->  no detectable difference
GATE: PASS

local-small-structured vs local-structured on eval_50, metric all_fields_right
  baseline right 29/50, candidate right 9/50, fixed 2, broke 22, p = 0.0  ->  B is worse
  broken cases: ML-0016, ML-0029, ML-0045, ML-0047, ML-0063, ML-0071, ML-0097, ML-0111, ML-0115, ML-0127, ML-0149, ML-0158, ML-0160, ML-0189, ML-0214, ML-0252, ML-0259, ML-0262, ML-0270, ML-0277, ML-0278, ML-0280
GATE: FAIL (significantly worse than the baseline; breaks 22 cases that the baseline solved (limit 2))
exit codes [0, 1]


**Result.** v2 passes. The 2B model fails with 22 broken entries. Wrong entries become draft cases for the next golden set.

In [13]:
from plant_assistant.data import load_records
from plant_assistant.evals.cases import cases_from_calls, save_cases

eval_records = {r.record_id: r for r in load_records() if r.record_id in {c.case_id for c in golden}}
drafts = cases_from_calls(RUNS_DIR / "course-v2-r1" / "results.jsonl", eval_records)
save_cases(drafts, RUNS_DIR / "cases_v2_draft.jsonl")
pd.DataFrame([{"case": d.case_id, "note": d.note, "status": d.status} for d in drafts])

,case,note,status
0,ML-0122-from-calls,wrong fields in course: work_type,draft


# Part B. The judge

## B1. Human labels

A person labelled the 213 unique summaries written by the 14 setups of practice 1. FAIL means the summary says something the entry does not support. The rules are in `evals/guideline_summary.md`.

In [14]:
print(pool["human"].value_counts().to_dict())
pool[pool["human"] == "FAIL"]["rule"].value_counts().to_frame("FAIL labels")

{'PASS': 181, 'FAIL': 32}


,FAIL labels
rule,
cause,17
added_fact,9
term,3
contradiction,3


**Result.** 32 of 213 summaries fail, mostly because of an invented cause.

## B2. Judges

- **Scale.** "You are an expert. Rate the helpfulness of this answer from 1 to 10." The judge sees only the summary. A score of 7 or more is PASS.
- **Binary.** One failure mode, the entry and the registry line in the prompt, the guideline rules, a reason and then PASS or FAIL.

Both prompts run on all 213 summaries with thinking off and on. The first 20 verdicts of the binary judge are below.

In [15]:
from plant_assistant.evals.analysis import load_judgements
from plant_assistant.evals.runs import pool_judges

pool_judges()  # 4 judges on 213 summaries, about 8 minutes on the first run
first20 = load_judgements("judge-binary-course")
pd.DataFrame([{"summary": row["summary"], "person": row["human"], "judge": first20[row["item_id"]]["verdict"],
               "reason": first20[row["item_id"]]["reason"]} for _, row in pool.head(20).iterrows()])

,summary,person,judge,reason
0,Blockage found causing failure,FAIL,FAIL,"The summary claims the blockage caused the failure, but the log entry only lists the failure and the block..."
1,Blockage found in the unit.,PASS,PASS,The summary accurately restates the defect (blockage) and its location (in the unit) as described in the l...
2,Blockage found in unit,PASS,PASS,The summary accurately restates the defect (blockage) and its location (in the unit) as explicitly stated ...
3,Blockage found after failure.,PASS,PASS,"The summary accurately restates the facts present in the log entry, which explicitly mentions a 'failure' ..."
4,blockage found,PASS,PASS,The summary accurately restates the specific defect mentioned in the log entry without adding unsupported ...
5,Blockage found in unit P-202A requiring repair.,PASS,PASS,"The summary accurately restates the defect (blockage) and the action (repair) for the specified unit, with..."
6,unit stopped due to blockage,FAIL,FAIL,"The summary claims a causal link ('due to') between the unit stopping and the blockage, which is not expli..."
7,Low insulation resistance detected,PASS,PASS,The summary accurately restates the defect 'low insulation resistance detected' which is explicitly mentio...
8,low insulation resistance,PASS,PASS,The summary accurately restates the specific defect mentioned in the log entry ('обнаружено низкое сопроти...
9,Low insulation resistance detected on equipment M-310.,PASS,PASS,The summary accurately restates the defect (low insulation resistance) and the equipment (M-310) as stated...


> **Predict.** Which judge agrees with the person best?

In [16]:
from plant_assistant.evals.analysis import judge_vs_human

JUDGES = ["judge-scale-course", "judge-scale-course-thinking", "judge-binary-course", "judge-binary-course-thinking"]
judge_vs_human(JUDGES)[["judge", "agreement, %", "kappa", "catches failures, %", "passes good answers, %"]]

,judge,"agreement, %",kappa,"catches failures, %","passes good answers, %"
0,judge-scale-course,23.5,0.02,96.9,10.5
1,judge-scale-course-thinking,15.5,0.00,100.0,0.6
2,judge-binary-course,93.4,0.72,68.8,97.8
3,judge-binary-course-thinking,95.3,0.82,87.5,96.7


In [17]:
# Scores of the scale judge
pd.Series([d["score"] for d in load_judgements("judge-scale-course").values()]).value_counts().sort_index().to_frame("summaries")

,summaries
1,113
2,80
10,20


**Result.** The binary judge with thinking reaches kappa 0.82 and catches 88% of failures. Without thinking it reaches 0.72 and 69%. The scale judge gives 1 or 2 to almost every short summary, so it catches all failures and has kappa 0.

## B3. Disagreements

In [18]:
best = load_judgements(f"judge-binary-{JUDGE}")
pool["judge"] = pool["item_id"].map(lambda i: (best.get(i) or {}).get("verdict"))
pool["judge reason"] = pool["item_id"].map(lambda i: (best.get(i) or {}).get("reason"))
pool[pool["judge"] != pool["human"]][["text", "summary", "human", "judge", "judge reason"]]

,text,summary,human,judge,judge reason
16,"FV305, внеплановый ремонт 30.06.2024, обнаружена коррозия седла, замеенна прокладка, простой, 6 hrs, Захар...",Corrosion of the saddle,FAIL,PASS,"The summary states only that corrosion of the saddle was found, which is explicitly stated in the log entry."
60,"Р-101: проведена чистка согласно WO-12147, в ходе работ обнаружен забитый фильтр, который был освобождён о...",Filter was clogged during cleaning operation.,PASS,FAIL,"The summary implies the filter became clogged during the cleaning operation, while the log only says a clo..."
109,"Транспортёр CV 401, отказ, ремонт по WO-57441 2024-11-16, проведена расцентровка с остановом, 8ч.",Conveyor misalignment requiring realignment.,PASS,FAIL,"The summary asserts a misalignment defect and a requirement for realignment, while the entry only states a..."
112,"Транспортёр CV 401, отказ, ремонт по WO-57441 2024-11-16, проведена расцентровка с остановом, 8ч.",misalignment requiring realignment,PASS,None,
113,"Транспортёр CV 401, отказ, ремонт по WO-57441 2024-11-16, проведена расцентровка с остановом, 8ч.",Misalignment,PASS,FAIL,"The entry only states that alignment/centering work was performed, not that the defect itself was misalign..."
115,"Транспортёр CV 401, отказ, ремонт по WO-57441 2024-11-16, проведена расцентровка с остановом, 8ч.",Conveyor CV-401 required realignment due to misalignment.,PASS,FAIL,"The summary adds a causal claim that the realignment was due to misalignment, which is not stated in the l..."
168,12.09.2026 P104 ремонт по заякве оператора греется масло заменено 4 ч без останова Смирнов Е.,Oil overheated; replaced parts.,PASS,FAIL,"The summary claims parts were replaced, but the log only states that oil was replaced."
169,12.09.2026 P104 ремонт по заякве оператора греется масло заменено 4 ч без останова Смирнов Е.,Oil overheated; replaced oil.,PASS,FAIL,"The entry states that the oil was heating up, not that it overheated, so the summary adds an unsupported s..."
184,E-140 20.02.24 WO-42901 остановлен утечка у уплотнения заменили прокладку 8 ч,seal leak at gasket,FAIL,PASS,"The entry states a leak at the seal and that a gasket was replaced, so the summary’s seal/gasket leak is s..."
198,E240 breakdown repair corroison gasket replaced 12 h,Corrosion on gasket,FAIL,PASS,"The entry mentions corrosion and a gasket replacement, which supports the summary's claim of corrosion on ..."


**Result.** 11 disagreements. Some are judge errors, some come from unclear rules, and one revealed two swapped human labels.

## B4. Few labels

An expert labels 40 summaries, the judge labels all of them. Prediction-powered inference (PPI) corrects the judge estimate with the 40 labels. The full labels give the true value for comparison.

In [19]:
from plant_assistant.evals.analysis import ppi_demo

ppi_demo(f"judge-binary-{JUDGE}", n_labelled=40)

,estimate,"faithful, %",low,high
0,expert on 40 only,82.5,70.6,94.4
1,judge on all 212,84.0,NaN,NaN
2,"PPI, judge on 172 and expert on 40",81.2,73.8,88.6
3,"truth, expert on all 212",84.9,NaN,NaN


**Result.** The expert alone gives 71–94%. PPI gives 74–89% and contains the true 84.9%.

# Conclusions

| Observation | Meaning |
|-------------|---------|
| v2 raised faithful summaries from 91–93% to 97–98% and broke nothing | measure the targeted field and check the rest |
| run 2 gave p = 0.063 | one run is one sample |
| 96–98% on `eval_50`, 80–84% on the holdout | a set used for tuning saturates |
| the scale judge catches 100% of failures and has kappa 0 | check both error rates of a judge |
| thinking raised the judge's catch rate from 69% to 88% | judge settings are measurable |
| a disagreement found a labelling error | read every disagreement |

# About the data

The data is synthetic and was built from the labels, so every entry is labelled automatically. Real data needs manual labelling. Synthetic test data has the traps from the lecture (block 3), such as easy cases, copied wording, one model family and polite simulated users.

1. Start from real logs, and add synthetic cases only to fill gaps.
2. Generate by dimensions with a different model family.
3. Remove duplicates, copied wording and easy cases, and let an expert check a sample.
4. Label with a written guideline and two people, and measure kappa.
5. Keep a holdout and add harder cases when the set saturates.

# Homework

1. Label 30 summaries with `evals/guideline_summary.md` and compute Cohen's kappa against `data/summary_labels.jsonl`.
2. Write a judge for the criterion "the summary describes the defect, not only the work". Label 30 cases and report kappa.
3. Write prompt v3 for the `work_type` error from A2 and compare it with v2 on the holdout with the paired test.